# MIIA Pothole Image Classification Challenge
**Objective**: Create a machine learning model to accurately predict the likelihood that an image contains a pothole.
**Metric**: Area Under the Curve (AUC)

This notebook is specifically designed for the Zindi MIIA Pothole challenge using `train_ids_labels.csv`, `test_ids_only.csv`, and `all_data.zip`.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Path Configuration and Data Extraction

In [ ]:
def setup_paths():
    base_dir = '/kaggle/input'
    train_csv = 'train_ids_labels.csv'
    test_csv = 'test_ids_only.csv'
    data_zip = 'all_data.zip'
    
    if os.path.exists(base_dir):
        for root, dirs, files in os.walk(base_dir):
            if 'train_ids_labels.csv' in files:
                train_csv = os.path.join(root, 'train_ids_labels.csv')
            if 'test_ids_only.csv' in files:
                test_csv = os.path.join(root, 'test_ids_only.csv')
            if 'all_data.zip' in files:
                data_zip = os.path.join(root, 'all_data.zip')
                
    return train_csv, test_csv, data_zip

TRAIN_CSV, TEST_CSV, DATA_ZIP = setup_paths()
WORKING_DIR = '/kaggle/working/images'

print(f"Train CSV: {TRAIN_CSV}")
print(f"Test CSV: {TEST_CSV}")
print(f"Data Zip: {DATA_ZIP}")

def extract_data(zip_path, extract_path):
    if not os.path.exists(extract_path):
        os.makedirs(extract_path, exist_ok=True)
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Extraction complete.")
    else:
        print("Data already extracted.")

if os.path.exists(DATA_ZIP):
    extract_data(DATA_ZIP, WORKING_DIR)
else:
    print("all_data.zip not found.")

## 2. Dataset Class

In [ ]:
class PotholeDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # The ID column might be 'id' or 'ID'
        img_id = self.df.iloc[idx].get('id') or self.df.iloc[idx].get('ID')
        # Check if extension is needed
        img_name = str(img_id)
        if not img_name.lower().endswith('.jpg'):
            img_name += '.jpg'
            
        img_path = os.path.join(self.img_dir, img_name)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            # Some datasets have images in subfolders
            found = False
            for root, _, files in os.walk(self.img_dir):
                if img_name in files:
                    image = Image.open(os.path.join(root, img_name)).convert("RGB")
                    found = True
                    break
            if not found:
                image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            return image, str(img_id)
        
        label = self.df.iloc[idx]['Label'] if 'Label' in self.df.columns else self.df.iloc[idx]['label']
        return image, torch.tensor(label, dtype=torch.float32)

## 3. Model Definition (ResNet50)

In [ ]:
def get_model():
    model = models.resnet50(pretrained=True)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    return model.to(device)

## 4. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=5):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    best_auc = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        # Validation
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                val_preds.extend(outputs.cpu().numpy())
                val_labels.extend(labels.numpy())
        
        auc = roc_auc_score(val_labels, val_preds)
        print(f"Epoch {epoch+1}, Train Loss: {train_loss/len(train_loader):.4f}, Val AUC: {auc:.4f}")
        
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), 'best_pothole_model.pth')
            
    return best_auc

## 5. Main Execution and Submission Generation

In [ ]:
if os.path.exists(TRAIN_CSV):
    df = pd.read_csv(TRAIN_CSV)
    train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
    
    transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    test_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_ds = PotholeDataset(train_df, WORKING_DIR, transform)
    val_ds = PotholeDataset(val_df, WORKING_DIR, test_transform)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
    
    model = get_model()
    train_model(model, train_loader, val_loader)
    
    # Inference on Test set
    if os.path.exists(TEST_CSV):
        test_df = pd.read_csv(TEST_CSV)
        test_ds = PotholeDataset(test_df, WORKING_DIR, test_transform, is_test=True)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
        
        model.load_state_dict(torch.load('best_pothole_model.pth'))
        model.eval()
        
        results = []
        with torch.no_grad():
            for images, ids in tqdm(test_loader, desc="Testing"):
                images = images.to(device)
                outputs = model(images)
                probs = outputs.cpu().numpy().flatten()
                for img_id, prob in zip(ids, probs):
                    results.append({'id': img_id, 'label': prob})
        
        sub_df = pd.DataFrame(results)
        sub_df.to_csv('submission.csv', index=False)
        print("Submission file generated: submission.csv")
        print(sub_df.head())
else:
    print("Train CSV not found. Ensure dataset is attached correctly.")